In [ ]:
## WORK

from __future__ import annotations

from pathlib import Path
from typing import Union, Literal, Any, Dict, List
import numpy as np
import pandas as pd


def build_rnk_from_deseq_xlsx_dir(
    input_dir: Union[str, Path],
    output_dir: Union[str, Path],
    *,
    metric: Literal["stat", "log2FoldChange"] = "log2FoldChange",
    sheet: Union[int, str] = 0,
    basemean_min: float = 1,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Generate GSEA-style .rnk files from a directory of DESeq2 Excel (.xlsx) results.

      - Reads all .xlsx files in `input_dir` (sorted).
      - Keeps only ["gene_name", "baseMean", <metric>] columns.
      - Cleans types:
          * gene_name as stripped string
          * baseMean and metric coerced to numeric
      - Filters out:
          * empty/"nan" gene names
          * non-finite metric values (NaN/inf)
          * rows with baseMean <= basemean_min
      - If a file becomes empty after filtering, it is skipped .
      - Deduplicates by gene_name keeping the row with max absolute metric (max |metric|).
      - Sorts by metric descending.
      - Writes one .rnk per input file in `output_dir`:
            gene_name \\t metric
        (no header, no index).

    Parameters
    
    input_dir : str or pathlib.Path
        Folder containing DESeq2 .xlsx files.
    output_dir : str or pathlib.Path
        Folder where .rnk files will be written (created if missing).
    metric : {"stat", "log2FoldChange"}, default="log2FoldChange"
        Which DESeq2 column to use as ranking score.
    sheet : int or str, default=0
        Excel sheet index (0-based) or sheet name.
    basemean_min : float, default=1
        Keep rows with baseMean > basemean_min.
    verbose : bool, default=True
        If True, prints progress and warnings.

    Returns
    
    pandas.DataFrame
        Summary table with one row per generated .rnk file.

    Examples
    --------
     # Example 1: rank by log2FoldChange
    >>> summary_lfc = build_rnk_from_deseq_xlsx_dir(
    ...     input_dir="/path/to/DESEQ_Unfiltered",
    ...     output_dir="/path/to/DESEQ_Unfiltered/InputFiltered",
    ...     metric="log2FoldChange",
    ...     basemean_min=1,
    ... )

    >>> # Example 2: rank by stat (Wald statistic)
    >>> summary_stat = build_rnk_from_deseq_xlsx_dir(
    ...     input_dir="/path/to/DESEQ_Unfiltered",
    ...     output_dir="/path/to/DESEQ_Unfiltered/InputFiltered_stat",
    ...     metric="stat",
    ...     basemean_min=1,
    ... )
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    xlsx_files = sorted(input_dir.glob("*.xlsx"))
    if not xlsx_files:
        raise FileNotFoundError(f"No .xlsx found in: {input_dir}")

    required_cols = ["gene_name", "baseMean", metric]
    summary: List[Dict[str, Any]] = []

    for fp in xlsx_files:
        df = pd.read_excel(fp, sheet_name=sheet, engine="openpyxl")

        missing = set(required_cols) - set(df.columns)
        if missing:
            raise KeyError(
                f"{fp.name} missing required columns: {sorted(missing)}. "
                f"Available: {list(df.columns)}"
            )

        df = df[required_cols].copy()

        # Clean types
        df["gene_name"] = df["gene_name"].astype(str).str.strip()
        df["baseMean"] = pd.to_numeric(df["baseMean"], errors="coerce")
        df[metric] = pd.to_numeric(df[metric], errors="coerce")

        # Filters: gene_name not empty/nan; metric finite
        df = df[(df["gene_name"] != "") & (df["gene_name"].str.lower() != "nan")]
        df = df[np.isfinite(df[metric])]

        # baseMean filter (strict >)
        df = df[df["baseMean"] > basemean_min]

        if df.empty:
            if verbose:
                print(f" {fp.name}: No genes after filters (Check baseMean/{metric}).")
            continue

        # Deduplicate per gene: keep max |metric|
        idx = df.groupby("gene_name")[metric].apply(lambda s: s.abs().idxmax())
        df = df.loc[idx].copy()

        # Sort by metric descending
        df = df.sort_values(metric, ascending=False)

        # Write .rnk: 2 columns, tab, no header
        out_path = output_dir / f"{fp.stem}.rnk"
        df[["gene_name", metric]].to_csv(out_path, sep="\t", header=False, index=False)

        summary.append(
            {
                "xlsx": fp.name,
                "rnk": out_path.name,
                "metric_used": metric,
                "genes_in_rnk": int(df.shape[0]),
                "baseMean_min_used": float(basemean_min),
                "metric_max": float(df[metric].max()),
                "metric_min": float(df[metric].min()),
            }
        )

        if verbose:
            print(f" {fp.name} → {out_path.name} ({df.shape[0]} geni)")

    return pd.DataFrame(summary)


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Union, Optional, Dict, List, Tuple, Any
import re
import shutil
import pandas as pd


def collect_latest_gsea_reports(
    gsea_root: Union[str, Path],
    *,
    collected_dir: Optional[Union[str, Path]] = None,
    pos_subdir: str = "Pos",
    neg_subdir: str = "Neg",
    glob_pattern: str = "gsea_report_for_na_*.tsv",
    pos_regex: str = r"gsea_report_for_na_pos_(\d+)\.tsv$",
    neg_regex: str = r"gsea_report_for_na_neg_(\d+)\.tsv$",
    manifest_name: str = "manifest_copied_reports.tsv",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Collect the latest GSEA report TSVs (pos/neg) from multiple run directories into a single folder.

    This function does:
      - Recursively finds all files matching `glob_pattern` under `gsea_root`.
      - Defines the "run directory" as the first directory under `gsea_root`
      - Splits files by direction using regex:
          * pos: `pos_regex`  (e.g., gsea_report_for_na_pos_<TS>.tsv)
          * neg: `neg_regex`  (e.g., gsea_report_for_na_neg_<TS>.tsv)
      - For each (run_dir, direction), selects the "best" candidate as:
          * maximum extracted timestamp from filename (integer after last underscore)
          * tie-breaker: most recent filesystem mtime
      - Copies the chosen file to:
          collected_dir / pos_subdir   (for pos)
          collected_dir / neg_subdir   (for neg)
        using a destination name:
          "<sanitized_run_dir>__<original_filename>"
      - Writes a manifest TSV with columns:
          run_dir, direction, source, dest
      - Returns the manifest as a DataFrame.

    Parameters
    ----------
    gsea_root : str or pathlib.Path
        Root directory containing GSEA run folders.
    collected_dir : str or pathlib.Path or None, optional
        Where to place collected reports. If None, defaults to `gsea_root / "CollectedReports"`.
    pos_subdir : str, default="Pos"
        Subdirectory under `collected_dir` for positive reports.
    neg_subdir : str, default="Neg"
        Subdirectory under `collected_dir` for negative reports.
    glob_pattern : str, default="gsea_report_for_na_*.tsv"
        Pattern used with rglob to discover candidate reports.
    pos_regex : str, default=r"gsea_report_for_na_pos_(\\d+)\\.tsv$"
        Regex that identifies positive reports and captures the timestamp.
    neg_regex : str, default=r"gsea_report_for_na_neg_(\\d+)\\.tsv$"
        Regex that identifies negative reports and captures the timestamp.
    manifest_name : str, default="manifest_copied_reports.tsv"
        Output filename for the manifest TSV (saved under `collected_dir`).
    verbose : bool, default=True
        If True, prints progress messages similar to the script.

    Returns
    -------
    pandas.DataFrame
        Manifest of copied reports.

    Examples
    --------
    >>> # Example: collect latest pos/neg reports per run directory
    >>> manifest = collect_latest_gsea_reports(
    ...     gsea_root="/Volumes/oXA/Thesis/Meth/Results/gsea/TAMR",
    ... )
    >>> print(manifest.head())

    >>> # Example: custom output location
    >>> manifest = collect_latest_gsea_reports(
    ...     gsea_root="/path/to/gsea/TAMR",
    ...     collected_dir="/path/to/gsea/TAMR/CollectedReports",
    ... )
    """
    gsea_root = Path(gsea_root)

    if collected_dir is None:
        collected_dir = gsea_root / "CollectedReports"
    collected_dir = Path(collected_dir)

    pos_dir = collected_dir / pos_subdir
    neg_dir = collected_dir / neg_subdir

    pos_dir.mkdir(parents=True, exist_ok=True)
    neg_dir.mkdir(parents=True, exist_ok=True)

    pos_re = re.compile(pos_regex)
    neg_re = re.compile(neg_regex)

    def sanitize(name: str) -> str:
        return re.sub(r"[^A-Za-z0-9._-]+", "_", name)

    def extract_ts(p: Path) -> int:
        m = re.search(r"_(\d+)\.tsv$", p.name)
        return int(m.group(1)) if m else -1

    # Find all candidate reports recursively
    all_tsv = list(gsea_root.rglob(glob_pattern))
    if verbose:
        print(f"Found {len(all_tsv)} reports {glob_pattern} under {gsea_root}")

    # Group by (run_dir_name, direction)
    groups: Dict[Tuple[str, str], List[Path]] = {}

    for p in all_tsv:
        try:
            run_dir_name = p.relative_to(gsea_root).parts[0]
        except Exception:
            # keep identical behavior: silently skip if relative_to fails
            continue

        if pos_re.search(p.name):
            groups.setdefault((run_dir_name, "pos"), []).append(p)
        elif neg_re.search(p.name):
            groups.setdefault((run_dir_name, "neg"), []).append(p)

    run_dirs = sorted({k[0] for k in groups.keys()})
    if verbose:
        print(f"Run directories with at least one report: {len(run_dirs)}")

    manifest_rows: List[Dict[str, Any]] = []

    # For each run dir and direction, choose best candidate and copy
    for run_dir_name in run_dirs:
        for direction in ["pos", "neg"]:
            candidates = groups.get((run_dir_name, direction), [])
            if not candidates:
                continue

            best = sorted(
                candidates,
                key=lambda p: (extract_ts(p), p.stat().st_mtime),
                reverse=True,
            )[0]

            run_label = sanitize(run_dir_name)
            dest_base = pos_dir if direction == "pos" else neg_dir
            dest_name = f"{run_label}__{best.name}"
            dest_path = dest_base / dest_name

            shutil.copy2(best, dest_path)

            manifest_rows.append(
                {
                    "run_dir": run_dir_name,
                    "direction": direction,
                    "source": str(best),
                    "dest": str(dest_path),
                }
            )

    if verbose:
        print(f"Copied Pos: {len(list(pos_dir.glob('*.tsv')))}")
        print(f"Copied Neg: {len(list(neg_dir.glob('*.tsv')))}")

    manifest = pd.DataFrame(manifest_rows)
    manifest_path = collected_dir / manifest_name
    manifest.to_csv(manifest_path, sep="\t", index=False)

    if verbose:
        print(f"Manifest: {manifest_path}")

    return manifest

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Union, Optional, Tuple
import pandas as pd
from itertools import combinations


def overlap_k_analysis(
    input_dir: Union[str, Path],
    *,
    out_dir: Optional[Union[str, Path]] = None,
    fdr_threshold: float = 0.25,
    pathway_col: str = "NAME",
    fdr_col: str = "FDR q-val",
    file_glob: str = "*.tsv",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Perform k-way overlap analysis of significant pathways across multiple GSEA TSV reports.

    This function mirrors the behavior of the provided code:
      - Reads all TSV files in `input_dir` matching `file_glob` (sorted).
      - For each file:
          * requires columns `pathway_col` and `fdr_col`
          * keeps pathways with FDR < `fdr_threshold`
          * stores them as a set keyed by the file stem
      - For each k = 2..N:
          * computes intersections for all combinations of k datasets
          * writes `common_pathways_k{k}.tsv` to `out_dir` if any overlap exists
          * prints a message if no overlap exists
      - Writes `summary_all_overlaps.tsv` to `out_dir` (always).
      - Returns the summary DataFrame.

    Parameters
    ----------
    input_dir : str or pathlib.Path
        Folder containing TSV reports (e.g., CollectedReports/Pos or CollectedReports/Neg).
    out_dir : str or pathlib.Path or None, optional
        Output directory. If None, defaults to `input_dir / "analysis"`.
    fdr_threshold : float, default=0.25
        Significance threshold used to select pathways (FDR < threshold).
    pathway_col : str, default="NAME"
        Column name containing pathway IDs/names.
    fdr_col : str, default="FDR q-val"
        Column name containing FDR values.
    file_glob : str, default="*.tsv"
        Glob pattern for selecting input files.
    verbose : bool, default=True
        If True, prints progress messages

    Returns
    -------
    pandas.DataFrame
        A summary of all non-empty overlaps across all k, with columns:
        - datasets (comma-separated stems)
        - k
        - n_common_pathways
        - pathways (semicolon-separated)

    Raises
    ------
    RuntimeError
        If no TSV files are found in `input_dir`.
    ValueError
        If required columns are missing in any file.

    Examples
    --------
    >>> # Example: run overlap analysis on Pos reports
    >>> pos_summary = overlap_k_analysis(
    ...     input_dir="/path/to/CollectedReports/Pos",
    ...     fdr_threshold=0.25,
    ... )
    >>> print(pos_summary.head())

    >>> # Example: run overlap analysis on Neg reports with a custom output dir
    >>> neg_summary = overlap_k_analysis(
    ...     input_dir="/path/to/CollectedReports/Neg",
    ...     out_dir="/path/to/CollectedReports/Neg/analysis_custom",
    ...     fdr_threshold=0.25,
    ... )
    """
    input_dir = Path(input_dir)

    if out_dir is None:
        out_dir = input_dir / "analysis"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(input_dir.glob(file_glob))
    if not files:
        raise RuntimeError(f"No .tsv files found in {input_dir}")

    if verbose:
        print(
            f"\nOverlap analysis on: {input_dir} | files: {len(files)} | FDR<{fdr_threshold}"
        )

    # Read significant pathways per dataset
    pathway_sets = {}
    for f in files:
        df = pd.read_csv(f, sep="\t", low_memory=False)

        missing = {pathway_col, fdr_col} - set(df.columns)
        if missing:
            raise ValueError(f"{f.name} is missing required columns: {missing}")

        sig = (
            df.loc[df[fdr_col] < fdr_threshold, pathway_col]
            .dropna()
            .astype(str)
            .unique()
        )
        pathway_sets[f.stem] = set(sig)

        if verbose:
            print(f"  {f.name}: {len(sig)} pathways (FDR < {fdr_threshold})")

    names = list(pathway_sets.keys())
    N = len(names)

    # Compute overlap by k
    summary_rows = []
    for k in range(2, N + 1):
        rows = []
        for combo in combinations(names, k):
            common = set.intersection(*(pathway_sets[n] for n in combo))
            if common:
                rows.append(
                    {
                        "datasets": ",".join(combo),
                        "k": k,
                        "n_common_pathways": len(common),
                        "pathways": ";".join(sorted(common)),
                    }
                )

        if rows:
            out_df = pd.DataFrame(rows)
            out_file = out_dir / f"common_pathways_k{k}.tsv"
            out_df.to_csv(out_file, sep="\t", index=False)
            summary_rows.extend(rows)

            if verbose:
                print(f"k={k}: saved {out_file} (rows={len(rows)})")
        else:
            if verbose:
                print(f"k={k}: no overlap")

    # Global summary
    summary = pd.DataFrame(summary_rows)
    summary_file = out_dir / "summary_all_overlaps.tsv"
    summary.to_csv(summary_file, sep="\t", index=False)

    if verbose:
        print(f"\nAnalysis completed. Output in: {out_dir}")

    return summary


def run_pos_neg_overlap(
    gsea_root: Union[str, Path],
    *,
    fdr_threshold: float = 0.25,
    pathway_col: str = "NAME",
    fdr_col: str = "FDR q-val",
    file_glob: str = "*.tsv",
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Convenience wrapper to run overlap_k_analysis on both Pos and Neg folders.

    It assumes this standard structure:
        gsea_root / "CollectedReports" / "Pos"
        gsea_root / "CollectedReports" / "Neg"

    Parameters
    ----------
    gsea_root : str or pathlib.Path
        Root folder that contains `CollectedReports/Pos` and `CollectedReports/Neg`.
    fdr_threshold, pathway_col, fdr_col, file_glob, verbose
        Passed through to `overlap_k_analysis`.

    Returns
    -------
    (pos_summary, neg_summary) : tuple[pandas.DataFrame, pandas.DataFrame]
        Two summary DataFrames (Pos first, then Neg).
    """
    gsea_root = Path(gsea_root)
    pos_input = gsea_root / "CollectedReports" / "Pos"
    neg_input = gsea_root / "CollectedReports" / "Neg"

    pos_summary = overlap_k_analysis(
        pos_input,
        fdr_threshold=fdr_threshold,
        pathway_col=pathway_col,
        fdr_col=fdr_col,
        file_glob=file_glob,
        verbose=verbose,
    )

    neg_summary = overlap_k_analysis(
        neg_input,
        fdr_threshold=fdr_threshold,
        pathway_col=pathway_col,
        fdr_col=fdr_col,
        file_glob=file_glob,
        verbose=verbose,
    )

    return pos_summary, neg_summary

In [ ]:
from pathlib import Path
import pandas as pd
from collections import defaultdict


def pathway_frequency(
    input_dir,
    fdr_threshold=0.25,
    pathway_col="NAME",
    fdr_col="FDR q-val",
    pattern="*.tsv",
    out_subdir="analysis",
    out_name="pathway_frequency.tsv",
    return_top=None,
):
    """
    Build a pathway frequency table from multiple GSEA report .tsv files.

    Parameters
    ----------
    input_dir : str | Path
        Directory containing GSEA report TSVs (e.g., CollectedReports/Pos or /Neg).
    fdr_threshold : float
        Keep pathways with FDR q-val < threshold for each dataset.
    pathway_col : str
        Column name containing pathway names (usually 'NAME').
    fdr_col : str
        Column name containing FDR values (usually 'FDR q-val').
    pattern : str
        Glob pattern for input files (default '*.tsv').
    out_subdir : str
        Subdirectory created inside input_dir to store outputs.
    out_name : str
        Output filename (TSV).
    return_top : int | None
        If set, returns only the top N rows (after sorting) in addition to saving full file.

    Returns
    -------
    freq_df : pd.DataFrame
        DataFrame with columns: pathway, n_datasets, datasets.
    out_path : Path
        Path to the saved TSV.
    """
    input_dir = Path(input_dir)
    files = sorted(input_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No '{pattern}' files found in {input_dir}")

    support = defaultdict(list)

    for f in files:
        df = pd.read_csv(f, sep="\t", low_memory=False)

        missing = {pathway_col, fdr_col} - set(df.columns)
        if missing:
            raise ValueError(f"{f.name} is missing required columns: {missing}")

        sig = (
            df.loc[df[fdr_col] < fdr_threshold, pathway_col]
            .dropna()
            .astype(str)
            .unique()
        )

        for p in sig:
            support[p].append(f.stem)

    freq_df = (
        pd.DataFrame(
            {
                "pathway": list(support.keys()),
                "n_datasets": [len(support[p]) for p in support],
                "datasets": [",".join(sorted(support[p])) for p in support],
            }
        )
        .sort_values(["n_datasets", "pathway"], ascending=[False, True])
    )

    out_dir = input_dir / out_subdir
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / out_name
    freq_df.to_csv(out_path, sep="\t", index=False)

    if return_top is not None:
        return freq_df.head(return_top), out_path

    return freq_df, out_path


# USAGE EXAMPLE:
# freq_df, out_path = pathway_frequency(
#     "/Volumes/oXA/Thesis/Meth/ResultsStat/gsea/TAMR/CollectedReports/Pos",
#     fdr_threshold=0.25,
#     return_top=20,
# )
# freq_df

In [ ]:
from pathlib import Path
import pandas as pd


def summarize_recurring_hallmarks(
    input_dir,
    fdr_threshold=0.25,
    min_support=3,
    pathway_col="NAME",
    nes_col="NES",
    fdr_col="FDR q-val",
    pattern="*.tsv",
    out_subdir="analysis",
    out_name="hallmark_summary.tsv",
):
    """
    Build a summary table of recurring pathways across multiple GSEA report TSVs.

    Parameters
    ----------
    input_dir : str | Path
        Folder containing per-dataset GSEA reports (e.g. CollectedReports/Pos or /Neg).
    fdr_threshold : float
        Significance cutoff (e.g. 0.25 for classic GSEA exploration).
    min_support : int
        Keep only pathways significant in at least this many datasets.
    pathway_col, nes_col, fdr_col : str
        Column names in GSEA report TSV.
    pattern : str
        Glob pattern for TSV files.
    out_subdir, out_name : str
        Where to write outputs inside input_dir.

    Returns
    -------
    summary_df : pd.DataFrame
        Summary table sorted by n_datasets desc, then median_NES desc.
    out_path : Path
        Saved TSV path.
    """
    input_dir = Path(input_dir)
    files = sorted(input_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No '{pattern}' files found in {input_dir}")

    rows = []
    for f in files:
        df = pd.read_csv(f, sep="\t", low_memory=False)

        missing = {pathway_col, nes_col, fdr_col} - set(df.columns)
        if missing:
            raise ValueError(
                f"{f.name} is missing required columns: {missing}. "
                f"Available columns: {list(df.columns)}"
            )

        df = df[[pathway_col, nes_col, fdr_col]].copy()
        df[pathway_col] = df[pathway_col].astype(str)

        # Keep only pathways significant in this dataset
        df = df[df[fdr_col] < fdr_threshold].dropna()

        if df.empty:
            continue

        dataset_name = f.stem  # e.g. BT474_TAMR_pos
        df["dataset"] = dataset_name
        rows.append(df)

    if not rows:
        raise RuntimeError(
            f"No significant pathways found with FDR<{fdr_threshold} in {input_dir}"
        )

    long_df = pd.concat(rows, ignore_index=True)

    # Aggregate per pathway
    summary_df = (
        long_df.groupby(pathway_col)
        .agg(
            n_datasets=("dataset", "nunique"),
            median_NES=(nes_col, "median"),
            min_FDR=(fdr_col, "min"),
            median_FDR=(fdr_col, "median"),
            datasets=("dataset", lambda s: ",".join(sorted(set(s)))),
        )
        .reset_index()
        .rename(columns={pathway_col: "pathway"})
    )

    # Filter by minimum support
    summary_df = summary_df[summary_df["n_datasets"] >= min_support].copy()

    # Sort
    summary_df = summary_df.sort_values(
        ["n_datasets", "median_NES", "min_FDR"],
        ascending=[False, False, True],
    )

    out_dir = input_dir / out_subdir
    out_dir.mkdir(exist_ok=True)
    out_path = out_dir / out_name
    summary_df.to_csv(out_path, sep="\t", index=False)

    return summary_df, out_path

In [ ]:
from pathlib import Path
from typing import Optional, Union, List, Dict, Tuple
import pandas as pd
import re
import difflib

# 
# Dataset label normalization
# 
def normalize_dataset_label(label: str) -> str:
    """
    Examples:
      'BT474_TAMR_pos' -> 'BT474_TAMR'
      'ZR75_1_TAM2_neg' -> 'ZR75_1_TAM2'
      'MCF7_TamR' -> 'MCF7_TamR'
    """
    s = str(label).strip()
    s = re.sub(r"__.*$", "", s)  # in case you have prefixes like "X__gsea_report..."
    s = re.sub(r"\.tsv$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"_(pos|neg)$", "", s, flags=re.IGNORECASE)
    return s


def pick_best_match(core: str, candidates: List[str], cutoff: float = 0.6) -> Optional[str]:
    """
    If there is no exact match, try fuzzy matching (difflib).
    """
    if core in candidates:
        return core
    # Try case-insensitive match
    lower_map = {c.lower(): c for c in candidates}
    if core.lower() in lower_map:
        return lower_map[core.lower()]

    matches = difflib.get_close_matches(core, candidates, n=1, cutoff=cutoff)
    return matches[0] if matches else None


# 
# Scan RunGsea: run dirs and my_analysis dirs
# 
def _extract_timestamp(name: str) -> int:
    """
    Extract the last long integer from the name (e.g. my_analysis....1767047432772 -> 1767047432772).
    If not present, returns -1.
    """
    m = re.findall(r"(\d{8,})", name)
    return int(m[-1]) if m else -1


def find_latest_analysis_dir(run_dir: Path) -> Optional[Path]:
    """
    Inside a run_dir, find the most recent 'my_analysis*' subdirectory.
    Supports any naming: my_analysis.GseaPreranked..., my_analysis..., etc.
    """
    subdirs = [d for d in run_dir.iterdir() if d.is_dir() and d.name.startswith("my_analysis")]
    if not subdirs:
        # Fallback: any dir that contains "my_analysis" in the name
        subdirs = [d for d in run_dir.iterdir() if d.is_dir() and "my_analysis" in d.name]

    if not subdirs:
        return None

    # Prefer timestamp (if present), otherwise mtime
    subdirs_sorted = sorted(
        subdirs,
        key=lambda p: (_extract_timestamp(p.name), p.stat().st_mtime),
        reverse=True,
    )
    return subdirs_sorted[0]


def build_run_index(gsea_root: Union[str, Path]) -> Dict[str, Dict[str, Union[str, Path]]]:
    """
    Scan gsea_root (RunGsea) and build an index:
      run_name -> {run_dir, analysis_dir}
    """
    gsea_root = Path(gsea_root)
    run_dirs = [d for d in gsea_root.iterdir() if d.is_dir()]

    index = {}
    for rd in run_dirs:
        ad = find_latest_analysis_dir(rd)
        index[rd.name] = {"run_dir": rd, "analysis_dir": ad}
    return index


# 
# Find HALLMARK file and leading-edge genes
# 
def find_pathway_tsv(
    index: Dict[str, Dict[str, Union[str, Path]]],
    dataset_label: str,
    pathway: str,
    fuzzy_cutoff: float = 0.6,
) -> Optional[Path]:
    """
    dataset_label: e.g. 'BT474_TAMR_pos'
    pathway: e.g. 'HALLMARK_MTORC1_SIGNALING'
    Search for: <analysis_dir>/<pathway>.tsv, otherwise rglob.
    """
    core = normalize_dataset_label(dataset_label)
    run_names = list(index.keys())

    best = pick_best_match(core, run_names, cutoff=fuzzy_cutoff)
    if best is None:
        return None

    analysis_dir = index[best]["analysis_dir"]
    if analysis_dir is None:
        return None
    analysis_dir = Path(analysis_dir)

    direct = analysis_dir / f"{pathway}.tsv"
    if direct.exists():
        return direct

    # Fallback: search recursively (in case of nested folders)
    matches = list(analysis_dir.rglob(f"{pathway}.tsv"))
    if not matches:
        return None

    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0]


def extract_leading_edge_genes(
    pathway_tsv: Union[str, Path],
    gene_col: str = "SYMBOL",
    core_col: str = "CORE ENRICHMENT",
) -> List[str]:
    """
    Leading-edge genes = rows where CORE ENRICHMENT == Yes, return SYMBOL.
    """
    pathway_tsv = Path(pathway_tsv)
    df = pd.read_csv(pathway_tsv, sep="\t", low_memory=False)

    missing = {gene_col, core_col} - set(df.columns)
    if missing:
        raise ValueError(
            f"{pathway_tsv.name} is missing columns {missing}. Available columns: {list(df.columns)}"
        )

    genes = (
        df.loc[df[core_col].astype(str).str.strip().str.lower().eq("yes"), gene_col]
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )
    return genes


# 
# Main: from summary -> leading edge per dataset/pathway + per-pathway frequency
# 
def extract_leading_edge_for_summary(
    summary_df: pd.DataFrame,
    gsea_root: Union[str, Path],
    out_dir: Union[str, Path],
    pathway_col: str = "pathway",
    datasets_col: str = "datasets",
    gene_col: str = "SYMBOL",
    core_col: str = "CORE ENRICHMENT",
    fuzzy_cutoff: float = 0.6,
    debug: bool = False,
) -> pd.DataFrame:
    """
    For each (pathway, dataset) in the summary:
      - find <pathway>.tsv in the corresponding run
      - extract leading-edge genes (CORE ENRICHMENT == Yes)
      - write:
          out_dir/leading_edge_by_dataset.tsv
          out_dir/by_dataset/<dataset>/<pathway>__leading_edge.txt
          out_dir/by_pathway/<pathway>__leading_edge_gene_frequency.tsv
          out_dir/leading_edge_pathway_summary.tsv
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    index = build_run_index(gsea_root)
    if debug:
        print(f"Run directories found in {gsea_root}: {len(index)}")
        # Print mapping run -> analysis_dir
        for rn, meta in index.items():
            print(f"  - {rn} -> {meta['analysis_dir']}")

    rows = []
    for _, r in summary_df.iterrows():
        pathway = str(r[pathway_col]).strip()
        datasets = [x.strip() for x in str(r[datasets_col]).split(",") if x.strip()]

        for ds in datasets:
            tsv_path = find_pathway_tsv(index, ds, pathway, fuzzy_cutoff=fuzzy_cutoff)

            if tsv_path is None:
                if debug:
                    print(f"[MISSING] dataset={ds} pathway={pathway}")
                rows.append(
                    {
                        "dataset": ds,
                        "dataset_core": normalize_dataset_label(ds),
                        "pathway": pathway,
                        "n_leading_edge": 0,
                        "genes": "",
                        "status": "MISSING_TSV",
                        "source_tsv": "",
                    }
                )
                continue

            genes = extract_leading_edge_genes(tsv_path, gene_col=gene_col, core_col=core_col)

            if debug:
                print(f"[FOUND] dataset={ds} pathway={pathway} genes={len(genes)}")

            rows.append(
                {
                    "dataset": ds,
                    "dataset_core": normalize_dataset_label(ds),
                    "pathway": pathway,
                    "n_leading_edge": len(genes),
                    "genes": ",".join(genes),
                    "status": "OK",
                    "source_tsv": str(tsv_path),
                }
            )

            ds_dir = out_dir / "by_dataset" / ds
            ds_dir.mkdir(parents=True, exist_ok=True)
            (ds_dir / f"{pathway}__leading_edge.txt").write_text(
                "\n".join(genes) + ("\n" if genes else ""),
                encoding="utf-8",
            )

    per_ds = pd.DataFrame(rows)
    per_ds.to_csv(out_dir / "leading_edge_by_dataset.tsv", sep="\t", index=False)

    # Per-pathway gene frequency
    ok = per_ds[per_ds["status"] == "OK"].copy()
    pw_dir = out_dir / "by_pathway"
    pw_dir.mkdir(exist_ok=True)

    pw_summary_rows = []
    for pathway, sub in ok.groupby("pathway"):
        gene_counts: Dict[str, int] = {}

        for genes_str in sub["genes"]:
            genes = [g for g in str(genes_str).split(",") if g]
            for g in set(genes):
                gene_counts[g] = gene_counts.get(g, 0) + 1

        if not gene_counts:
            continue

        freq_df = (
            pd.DataFrame(
                {"gene": list(gene_counts.keys()), "n_datasets": list(gene_counts.values())}
            ).sort_values(["n_datasets", "gene"], ascending=[False, True])
        )
        freq_df.to_csv(pw_dir / f"{pathway}__leading_edge_gene_frequency.tsv", sep="\t", index=False)

        pw_summary_rows.append(
            {
                "pathway": pathway,
                "n_datasets": int(sub["dataset"].nunique()),
                "n_unique_leading_edge_genes": int(freq_df.shape[0]),
                "top_gene": freq_df.iloc[0]["gene"],
                "top_gene_n_datasets": int(freq_df.iloc[0]["n_datasets"]),
            }
        )

    pw_summary = pd.DataFrame(pw_summary_rows)
    if pw_summary.empty:
        pw_summary = pd.DataFrame(
            columns=[
                "pathway",
                "n_datasets",
                "n_unique_leading_edge_genes",
                "top_gene",
                "top_gene_n_datasets",
            ]
        )
    else:
        pw_summary = pw_summary.sort_values(["n_datasets", "pathway"], ascending=[False, True])

    pw_summary.to_csv(out_dir / "leading_edge_pathway_summary.tsv", sep="\t", index=False)

    if debug:
        print("\nSTATUS COUNTS:")
        print(per_ds["status"].value_counts(dropna=False).to_string())

    return per_ds

In [ ]:
from pathlib import Path
import pandas as pd


def build_consensus_leading_edge_genes(
    leading_edge_root,
    min_datasets=2,
    in_subdir="by_pathway",
    pattern="*__leading_edge_gene_frequency.tsv",
    out_subdir="consensus_min2",
    save_wide=True,
):
    """
    Build consensus leading-edge gene lists per pathway using a minimum dataset-support threshold.

    Parameters
    ----------
    leading_edge_root : str | Path
        Directory that contains 'by_pathway/' with the frequency TSVs.
    min_datasets : int
        Keep genes with n_datasets >= this threshold (default=2).
    in_subdir : str
        Input subdir name (default 'by_pathway').
    pattern : str
        Glob pattern for frequency files.
    out_subdir : str
        Output subdir name where consensus results will be written.
    save_wide : bool
        If True, also saves a "wide" table with genes collapsed into a single string.

    Returns
    -------
    long_df : pd.DataFrame
        Long table: pathway, gene, n_datasets
    wide_df : pd.DataFrame (or None)
        Wide table: pathway, n_genes, genes (collapsed string)
    out_dir : Path
        Output directory containing saved files.
    """
    leading_edge_root = Path(leading_edge_root)
    in_dir = leading_edge_root / in_subdir
    if not in_dir.exists():
        raise FileNotFoundError(f"Input dir not found: {in_dir}")

    files = sorted(in_dir.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files found with pattern {pattern} in {in_dir}")

    out_dir = leading_edge_root / out_subdir
    out_dir.mkdir(parents=True, exist_ok=True)

    long_rows = []
    wide_rows = []

    for fp in files:
        # Pathway name is the filename before "__leading_edge_gene_frequency.tsv"
        pathway = fp.name.replace("__leading_edge_gene_frequency.tsv", "")

        df = pd.read_csv(fp, sep="\t")
        missing = {"gene", "n_datasets"} - set(df.columns)
        if missing:
            raise ValueError(
                f"{fp.name} is missing required columns: {missing}. "
                f"Available columns: {list(df.columns)}"
            )

        df["n_datasets"] = pd.to_numeric(df["n_datasets"], errors="coerce")
        df = df.dropna(subset=["gene", "n_datasets"])
        df = df[df["n_datasets"] >= min_datasets].copy()

        # Save per-pathway
        per_pathway_out = out_dir / f"{pathway}__consensus_min{min_datasets}.tsv"
        df.sort_values(["n_datasets", "gene"], ascending=[False, True]).to_csv(
            per_pathway_out, sep="\t", index=False
        )

        # Accumulate long
        for _, r in df.iterrows():
            long_rows.append(
                {
                    "pathway": pathway,
                    "gene": str(r["gene"]),
                    "n_datasets": int(r["n_datasets"]),
                }
            )

        # Accumulate wide
        genes = (
            df.sort_values(["n_datasets", "gene"], ascending=[False, True])["gene"]
            .astype(str)
            .tolist()
        )
        wide_rows.append({"pathway": pathway, "n_genes": len(genes), "genes": ",".join(genes)})

    long_df = pd.DataFrame(long_rows).sort_values(
        ["pathway", "n_datasets", "gene"], ascending=[True, False, True]
    )
    long_df.to_csv(
        out_dir / f"consensus_all_pathways_long_min{min_datasets}.tsv",
        sep="\t",
        index=False,
    )

    wide_df = None
    if save_wide:
        wide_df = pd.DataFrame(wide_rows).sort_values(["n_genes", "pathway"], ascending=[False, True])
        wide_df.to_csv(
            out_dir / f"consensus_all_pathways_wide_min{min_datasets}.tsv",
            sep="\t",
            index=False,
        )

    return long_df, wide_df, out_dir

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Union, Optional, Sequence, Tuple, Dict, Any, List
import re

import numpy as np
import pandas as pd


def process_methylation_gene_level_promoter(
    input_dir: Union[str, Path],
    *,
    out_dir: Optional[Union[str, Path]] = None,
    # Raw (human-readable) column names you expect in the input files
    gene_col_raw: str = "Gene Name",
    annot_col_raw: str = "annotation intuitive",
    fdr_col_raw: str = "adj.P.Val",
    db_col_raw: str = "deltaBeta",
    # Biology/thresholds
    delta_cutoff: float = 0.20,
    promoter_keywords: Sequence[str] = ("promoter", "tss"),
    # I/O
    patterns: Sequence[str] = ("*.txt", "*.tsv"),
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, Path]:
    """
    Convert per-comparison methylation tables (CpG/DMP-level) into gene-level promoter summaries.

    This function does:
      - Scans `input_dir` for input files matching `patterns` (default: *.txt and *.tsv).
      - For each file:
          * Reads as TSV (`sep="\\t"`).
          * Locates required columns by "normalized" matching of `*_col_raw` names:
              - gene column (e.g., "Gene Name")
              - annotation column (e.g., "annotation intuitive")
              - FDR column (e.g., "adj.P.Val")
              - deltaBeta column (e.g., "deltaBeta")
          * Keeps only those columns and renames them to:
              - gene_raw, annotation, adjP, deltaBeta
          * Converts adjP and deltaBeta to numeric (coerce errors to NaN).
          * Drops rows with missing gene_raw/annotation/adjP/deltaBeta.
          * Splits rows with multiple genes (separators: ; , | /) and explodes to one gene per row.
          * Filters promoter-only rows where `annotation` contains any `promoter_keywords`
            (case-insensitive).
          * If no promoter rows exist, records a status row and skips output for that file.
          * For promoter rows:
              - Computes abs(deltaBeta)
              - Aggregates per gene:
                  n_promoter_DMP = number of promoter DMPs for the gene
                  min_FDR_promoter = minimum adjP across promoter DMPs
                  max_abs_deltaBeta_promoter = maximum |deltaBeta| across promoter DMPs
              - Selects a representative CpG/DMP per gene: the one with maximum |deltaBeta|
                (deltaBeta_rep, adjP_rep).
              - Assigns `promoter_status` from deltaBeta_rep:
                  hyper if deltaBeta_rep >= delta_cutoff
                  hypo  if deltaBeta_rep <= -delta_cutoff
                  small otherwise
              - Adds `comparison` column = file stem.
              - Writes per-comparison output:
                  <out_dir>/<stem>__gene_promoter.tsv
      - Writes a processing summary:
          <out_dir>/SUMMARY_processing.tsv
      - Writes a combined table across all comparisons (if any):
          <out_dir>/ALL_COMPARISONS__gene_promoter.tsv
      - Returns:
          (combined_df, summary_df, out_dir_path)

    Parameters
    ----------
    input_dir : str or pathlib.Path
        Directory containing per-comparison methylation result files in TSV format
        (typically exported as .txt or .tsv).
    out_dir : str or pathlib.Path or None, optional
        Output directory. If None, defaults to `input_dir / "gene_level_promoter"`.
    gene_col_raw : str, default="Gene Name"
        Expected (human) name of the gene column; matched robustly by normalized substring logic.
    annot_col_raw : str, default="annotation intuitive"
        Expected name of the genomic annotation column.
    fdr_col_raw : str, default="adj.P.Val"
        Expected name of adjusted p-value/FDR column.
    db_col_raw : str, default="deltaBeta"
        Expected name of delta-beta column.
    delta_cutoff : float, default=0.20
        Absolute deltaBeta cutoff for calling a gene hyper/hypo (based on representative CpG).
    promoter_keywords : Sequence[str], default=("promoter", "tss")
        Keywords used to label promoter rows based on the annotation column (case-insensitive).
    patterns : Sequence[str], default=("*.txt","*.tsv")
        File patterns to search for in `input_dir`.
    verbose : bool, default=True
        If True, prints a compact completion message and status counts similar to the script.

    Returns
    -------
    combined_df : pandas.DataFrame
        Combined per-gene promoter table across all processed comparisons.
        Empty if no per-comparison outputs were created.
    summary_df : pandas.DataFrame
        Per-file processing summary with counts and status.
    out_dir_path : pathlib.Path
        Output directory path where files were written.

    Raises
    ------
    FileNotFoundError
        If no input files matching `patterns` are found.
    ValueError
        If any required columns cannot be found in a given file.

    Examples
    --------
    >>> # Example: build gene-level promoter methylation summaries for all comparisons
    >>> combined_df, summary_df, out_dir = process_methylation_gene_level_promoter(
    ...     input_dir="/Users/alessandrogiordano/Desktop/TesiDottorato/Meth/Array/TAMR",
    ...     delta_cutoff=0.20,
    ...     promoter_keywords=("promoter", "tss"),
    ... )
    >>> print(out_dir)
    >>> print(summary_df["status"].value_counts())
    >>> print(combined_df.head())
    """
    input_dir = Path(input_dir)
    if out_dir is None:
        out_dir = input_dir / "gene_level_promoter"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 
    # Utils (kept local to preserve "script-like" behavior)
    # 
    def _normalize_colname(s: str) -> str:
        return re.sub(r"[\s_\.\-]+", "", str(s).strip().lower())

    def _find_col(df_cols, target_raw: str) -> Optional[str]:
        target = _normalize_colname(target_raw)
        norm_map = {_normalize_colname(c): c for c in df_cols}

        if target in norm_map:
            return norm_map[target]

        matches: List[str] = []
        for c in df_cols:
            cn = _normalize_colname(c)
            if target in cn:
                matches.append(c)

        if not matches:
            return None

        matches = sorted(matches, key=lambda x: len(str(x)))
        return matches[0]

    def _is_promoter(annotation: str) -> bool:
        a = str(annotation).lower()
        return any(k.lower() in a for k in promoter_keywords)

    def _explode_genes(series: pd.Series) -> pd.Series:
        """
        Split on common separators (;/,|/) and return list-like entries (for explode()).
        """
        s = series.astype(str).str.strip()
        s = s.replace({"nan": np.nan, "": np.nan})
        return s.str.split(r"[;,|/]+", regex=True)

    # 
    # Discover input files
    # 
    files: List[Path] = []
    for pat in patterns:
        files.extend(list(input_dir.glob(pat)))
    files = sorted(files)

    if not files:
        raise FileNotFoundError(f"No files matching {list(patterns)} found in {input_dir}")

    all_long_rows: List[pd.DataFrame] = []
    summary_rows: List[Dict[str, Any]] = []

    # 
    # Process each file
    # 
    for fp in files:
        df = pd.read_csv(fp, sep="\t", low_memory=False)

        gene_col = _find_col(df.columns, gene_col_raw)
        annot_col = _find_col(df.columns, annot_col_raw)
        fdr_col = _find_col(df.columns, fdr_col_raw)
        db_col = _find_col(df.columns, db_col_raw)

        missing = [("Gene", gene_col), ("Annot", annot_col), ("FDR", fdr_col), ("deltaBeta", db_col)]
        missing = [name for name, col in missing if col is None]
        if missing:
            raise ValueError(
                f"{fp.name}: required columns not found: {missing}\n"
                f"Available columns:\n{list(df.columns)}"
            )

        df = df[[gene_col, annot_col, fdr_col, db_col]].copy()
        df.columns = ["gene_raw", "annotation", "adjP", "deltaBeta"]

        df["adjP"] = pd.to_numeric(df["adjP"], errors="coerce")
        df["deltaBeta"] = pd.to_numeric(df["deltaBeta"], errors="coerce")

        df = df.dropna(subset=["gene_raw", "annotation", "deltaBeta", "adjP"])
        if verbose:
            n_bad = int((df["adjP"] > 0.05).sum())
            if n_bad > 0:
                max_bad = float(df.loc[df["adjP"] > 0.05, "adjP"].max())
                print(
                    f" {fp.name}: found {n_bad} rows with adjP > 0.05 (max={max_bad:.3g}). "
                    f"Input may not be filtered by champ.DMP."
                )
        df["gene_raw"] = df["gene_raw"].astype(str).str.strip()

        # Explode multiple genes per row
        df["gene_list"] = _explode_genes(df["gene_raw"])
        df = df.explode("gene_list")
        df["gene"] = df["gene_list"].astype(str).str.strip()
        df = df[(df["gene"] != "") & (df["gene"].str.lower() != "nan")]

        # promoter-only
        prom = df[df["annotation"].apply(_is_promoter)].copy()

        if prom.empty:
            summary_rows.append(
                {
                    "file": fp.name,
                    "comparison": fp.stem,
                    "n_rows_total": int(df.shape[0]),
                    "n_rows_promoter": 0,
                    "n_genes_promoter": 0,
                    "status": "NO_PROMOTER_ROWS",
                }
            )
            continue

        prom["absDB"] = prom["deltaBeta"].abs()

        # Representative CpG/DMP per gene: max |deltaBeta|
        idx = prom.groupby("gene")["absDB"].idxmax()
        rep = prom.loc[idx, ["gene", "deltaBeta", "adjP"]].rename(
            columns={"deltaBeta": "deltaBeta_rep", "adjP": "adjP_rep"}
        )

        # Aggregate per gene
        agg = (
            prom.groupby("gene")
            .agg(
                n_promoter_DMP=("gene", "size"),
                min_FDR_promoter=("adjP", "min"),
                max_abs_deltaBeta_promoter=("absDB", "max"),
            )
            .reset_index()
        )

        out = agg.merge(rep, on="gene", how="left")

        def _status(db: float) -> str:
            if pd.isna(db):
                return "none"
            if db >= delta_cutoff:
                return "hyper"
            if db <= -delta_cutoff:
                return "hypo"
            return "small"

        out["promoter_status"] = out["deltaBeta_rep"].apply(_status)
        out.insert(0, "comparison", fp.stem)

        out_path = out_dir / f"{fp.stem}__gene_promoter.tsv"
        out.to_csv(out_path, sep="\t", index=False)

        all_long_rows.append(out)

        summary_rows.append(
            {
                "file": fp.name,
                "comparison": fp.stem,
                "n_rows_total": int(df.shape[0]),
                "n_rows_promoter": int(prom.shape[0]),
                "n_genes_promoter": int(out.shape[0]),
                "status": "OK",
                "out_file": out_path.name,
            }
        )

    # 
    # Write global outputs
    # 
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(out_dir / "SUMMARY_processing.tsv", sep="\t", index=False)

    if all_long_rows:
        combined_df = pd.concat(all_long_rows, ignore_index=True)
        combined_df.to_csv(out_dir / "ALL_COMPARISONS__gene_promoter.tsv", sep="\t", index=False)
    else:
        combined_df = pd.DataFrame()

    if verbose:
        print("Done.")
        print("Output dir:", out_dir)
        if not summary_df.empty and "status" in summary_df.columns:
            print("\nStatus counts:")
            print(summary_df["status"].value_counts(dropna=False).to_string())

    return combined_df, summary_df, out_dir

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Union, Optional, Dict, Tuple, List, Any
import numpy as np
import pandas as pd


def integrate_gsea_consensus_with_methylation(
    *,
    meth_all_tsv: Union[str, Path],
    gsea_le_root: Union[str, Path],
    map_gsea_to_meth: Dict[str, str],
    out_dir: Optional[Union[str, Path]] = None,
    min_support: int = 2,
    # consensus sources
    consensus_long_name: str = "consensus_all_pathways_long_min2.tsv",
    by_pathway_subdir: str = "by_pathway",
    freq_suffix: str = "__leading_edge_gene_frequency.tsv",
    # methylation columns (expected from your gene_level_promoter outputs)
    meth_gene_col: str = "gene",
    meth_comp_col: str = "comparison",
    meth_status_col: str = "promoter_status",
    # outputs
    summary_out_name: Optional[str] = None,
    hits_out_name: Optional[str] = None,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, Path]:
    """
    Integrate GSEA leading-edge consensus genes (per pathway) with gene-level promoter methylation summaries.

    This function does:
      1) Builds a consensus gene set per pathway using leading-edge gene frequency files:
         - Prefer an existing consensus-long TSV if found under `gsea_le_root` (rglob).
         - Otherwise reconstruct consensus from `gsea_le_root/by_pathway/*__leading_edge_gene_frequency.tsv`.
         - Keeps genes with n_datasets >= `min_support`.
      2) Loads methylation gene-level promoter table from `meth_all_tsv`
         (e.g., ALL_COMPARISONS__gene_promoter.tsv), normalizes gene symbols to uppercase,
         and indexes rows by methylation comparison.
      3) For each (gsea_dataset -> methylation_comparison) mapping and each pathway:
         - Intersects methylation genes with the pathway consensus gene set
         - Computes counts (hyper/hypo/small), summary metrics, and gene lists
         - Optionally stores gene-level hit rows for drill-down
      4) Writes two TSV outputs:
         - INTEGRATION__dataset_pathway_summary_min{min_support}.tsv
         - INTEGRATION__gene_level_hits_min{min_support}.tsv

    Parameters
    ----------
    meth_all_tsv : str or pathlib.Path
        Path to the combined methylation table produced by
        `process_methylation_gene_level_promoter()`, usually:
        .../gene_level_promoter/ALL_COMPARISONS__gene_promoter.tsv
    gsea_le_root : str or pathlib.Path
        Root directory for leading-edge outputs (e.g. LeadingEdge_Neg or LeadingEdge_Pos).
        Must contain `by_pathway/` with frequency files, and may contain an existing
        `consensus_all_pathways_long_min2.tsv` (or similar) somewhere under this root.
    map_gsea_to_meth : dict[str, str]
        Mapping from GSEA dataset label (e.g. "MCF7_TamR_neg") to methylation comparison
        name (e.g. "MCF7_TAMR_vs_MCF7WT"). This is the key bridge between the two worlds.
    out_dir : str or pathlib.Path or None, optional
        Output directory. If None, defaults to:
        gsea_le_root / f"integration_with_methylation_min{min_support}"
    min_support : int, default=2
        Minimum dataset support for a gene to be considered "consensus" within a pathway.
    consensus_long_name : str, default="consensus_all_pathways_long_min2.tsv"
        Filename to look for (via rglob) under `gsea_le_root` as a precomputed consensus-long table.
        If found, it is used; otherwise consensus is reconstructed from frequency files.
        Note: even if you pass a min_support != 2, the function still filters by `min_support`.
    by_pathway_subdir : str, default="by_pathway"
        Subdirectory under `gsea_le_root` containing per-pathway gene frequency files.
    freq_suffix : str, default="__leading_edge_gene_frequency.tsv"
        Suffix used to identify frequency files.
    meth_gene_col, meth_comp_col, meth_status_col : str
        Column names expected in the methylation table.
    summary_out_name : str or None, optional
        Custom filename for the dataset×pathway summary TSV. If None, uses default naming.
    hits_out_name : str or None, optional
        Custom filename for gene-level hits TSV. If None, uses default naming.
    verbose : bool, default=True
        If True, prints concise progress messages.

    Returns
    -------
    summary_df : pandas.DataFrame
        Dataset × pathway summary with counts and summary statistics.
    gene_hits_df : pandas.DataFrame
        Gene-level methylation hits for each dataset × pathway (drill-down table).
    out_dir_path : pathlib.Path
        Output directory where TSV files were written.

    Examples
    --------
    >>> map_gsea_to_meth = {
    ...     "BT474_TAMR_neg": "BT474_TAM1_to_BT474wt",
    ...     "MCF7_TamR_neg": "MCF7_TAMR_vs_MCF7WT",
    ... }
    >>> summary_df, hits_df, out_dir = integrate_gsea_consensus_with_methylation(
    ...     meth_all_tsv="/path/to/gene_level_promoter/ALL_COMPARISONS__gene_promoter.tsv",
    ...     gsea_le_root="/path/to/LeadingEdge_Neg",
    ...     map_gsea_to_meth=map_gsea_to_meth,
    ...     min_support=2,
    ... )
    >>> print(out_dir)
    >>> print(summary_df.head())
    """
    meth_all_tsv = Path(meth_all_tsv)
    gsea_le_root = Path(gsea_le_root)

    by_pathway_dir = gsea_le_root / by_pathway_subdir
    if out_dir is None:
        out_dir = gsea_le_root / f"integration_with_methylation_min{min_support}"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 
    # 1) Consensus genes per pathway (min_support)
    # 
    consensus_long = None
    found = list(gsea_le_root.rglob(consensus_long_name))
    if found:
        consensus_long_path = found[0]
        consensus_long = pd.read_csv(consensus_long_path, sep="\t")
        consensus_long = consensus_long[
            pd.to_numeric(consensus_long["n_datasets"], errors="coerce") >= min_support
        ].copy()
    else:
        freq_files = sorted(by_pathway_dir.glob(f"*{freq_suffix}"))
        if not freq_files:
            raise FileNotFoundError(f"No frequency files found in {by_pathway_dir}")

        rows: List[Dict[str, Any]] = []
        for fp in freq_files:
            pathway = fp.name.replace(freq_suffix, "")
            df = pd.read_csv(fp, sep="\t")
            if not {"gene", "n_datasets"}.issubset(df.columns):
                raise ValueError(
                    f"{fp.name} must contain columns {{'gene','n_datasets'}}. Columns: {list(df.columns)}"
                )

            df["n_datasets"] = pd.to_numeric(df["n_datasets"], errors="coerce")
            df = df.dropna(subset=["gene", "n_datasets"])
            df = df[df["n_datasets"] >= min_support]
            for _, r in df.iterrows():
                rows.append(
                    {
                        "pathway": pathway,
                        "gene": str(r["gene"]),
                        "n_datasets": int(r["n_datasets"]),
                    }
                )
        consensus_long = pd.DataFrame(rows)

    if consensus_long.empty:
        raise RuntimeError(f"Consensus table is empty after filtering min_support={min_support}.")

    consensus_long["gene_norm"] = consensus_long["gene"].astype(str).str.strip().str.upper()

    pathway_to_genes = {
        pw: set(sub["gene_norm"].tolist()) for pw, sub in consensus_long.groupby("pathway")
    }

    if verbose:
        print(f"Consensus pathways (min{min_support}): {len(pathway_to_genes)}")

    # 
    # 2) Load methylation (gene-level promoter)
    # 
    meth = pd.read_csv(meth_all_tsv, sep="\t", low_memory=False)
    if meth_gene_col not in meth.columns or meth_comp_col not in meth.columns:
        raise ValueError(
            f"Methylation table is missing required columns: "
            f"{[c for c in [meth_gene_col, meth_comp_col] if c not in meth.columns]}. "
            f"Columns: {list(meth.columns)}"
        )

    meth["gene_norm"] = meth[meth_gene_col].astype(str).str.strip().str.upper()
    meth[meth_comp_col] = meth[meth_comp_col].astype(str).str.strip()

    meth_by_comp = {c: df.copy() for c, df in meth.groupby(meth_comp_col)}

    # 
    # 3) Integration
    # 
    summary_rows: List[Dict[str, Any]] = []
    gene_hits_rows: List[pd.DataFrame] = []

    for gsea_ds, comp in map_gsea_to_meth.items():
        if comp not in meth_by_comp:
            if verbose:
                print(f"Methylation comparison not found for {gsea_ds} -> {comp}")
            continue

        mdf = meth_by_comp[comp]

        for pathway, geneset in pathway_to_genes.items():
            sub = mdf[mdf["gene_norm"].isin(geneset)].copy()

            n_consensus = len(geneset)
            n_found = int(sub.shape[0])

            if meth_status_col in sub.columns and n_found:
                n_hyper = int((sub[meth_status_col] == "hyper").sum())
                n_hypo = int((sub[meth_status_col] == "hypo").sum())
                n_small = int((sub[meth_status_col] == "small").sum())
            else:
                n_hyper = n_hypo = n_small = 0

            hyper_genes = (
                ",".join(sorted(sub.loc[sub[meth_status_col] == "hyper", "gene_norm"].unique()))
                if (n_found and meth_status_col in sub.columns)
                else ""
            )
            hypo_genes = (
                ",".join(sorted(sub.loc[sub[meth_status_col] == "hypo", "gene_norm"].unique()))
                if (n_found and meth_status_col in sub.columns)
                else ""
            )

            med_abs_db = (
                float(np.nanmedian(sub.get("max_abs_deltaBeta_promoter", np.array([]))))
                if n_found
                else np.nan
            )
            min_fdr = (
                float(np.nanmin(sub.get("min_FDR_promoter", np.array([]))))
                if n_found
                else np.nan
            )

            summary_rows.append(
                {
                    "gsea_dataset": gsea_ds,
                    "methylation_comparison": comp,
                    "pathway": pathway,
                    "n_consensus_genes": int(n_consensus),
                    "n_genes_with_promoter_DMP": int(n_found),
                    "n_hyper": int(n_hyper),
                    "n_hypo": int(n_hypo),
                    "n_small": int(n_small),
                    "median_maxAbsDeltaBeta_promoter": med_abs_db,
                    "best_minFDR_promoter": min_fdr,
                    "hyper_genes": hyper_genes,
                    "hypo_genes": hypo_genes,
                }
            )

            if n_found:
                sub2 = sub.copy()
                sub2.insert(0, "gsea_dataset", gsea_ds)
                sub2.insert(1, "methylation_comparison", comp)
                sub2.insert(2, "pathway", pathway)

                keep = [
                    "gsea_dataset",
                    "methylation_comparison",
                    "pathway",
                    "gene",
                    "gene_norm",
                    "promoter_status",
                    "n_promoter_DMP",
                    "min_FDR_promoter",
                    "max_abs_deltaBeta_promoter",
                    "deltaBeta_rep",
                    "adjP_rep",
                ]
                keep = [c for c in keep if c in sub2.columns]
                gene_hits_rows.append(sub2[keep])

    summary_df = pd.DataFrame(summary_rows)
    gene_hits_df = pd.concat(gene_hits_rows, ignore_index=True) if gene_hits_rows else pd.DataFrame()

    # 
    # Write outputs
    # 
    if summary_out_name is None:
        summary_out_name = f"INTEGRATION__dataset_pathway_summary_min{min_support}.tsv"
    if hits_out_name is None:
        hits_out_name = f"INTEGRATION__gene_level_hits_min{min_support}.tsv"

    summary_path = out_dir / summary_out_name
    hits_path = out_dir / hits_out_name

    summary_df.to_csv(summary_path, sep="\t", index=False)
    gene_hits_df.to_csv(hits_path, sep="\t", index=False)

    if verbose:
        print("Saved:")
        print(" -", summary_path)
        print(" -", hits_path)

    return summary_df, gene_hits_df, out_dir


MAP_EXPR_TO_METH = {
    "risultati_geni_unfiltered_BT474_TAMR_treatment_run2.xlsx": "BT474_TAM1_to_BT474wt",
    "risultati_geni_unfiltered_MCF7_TamR_treatment_run2.xlsx": "MCF7_TAMR_vs_MCF7WT",
    "risultati_geni_unfiltered_MCF7_TAMR4_treatment_run2.xlsx": "MCF7_TAMR4_vs_MCF7_S05_WT",
    "risultati_geni_unfiltered_T47D_TAMR_treatment_run2.xlsx": "T47D_TAM2_vs_T47Dwt",
    "risultati_geni_unfiltered_T47D_TR1_treatment_run2.xlsx": "T47D_TR1_vs_T47D_S2wt",
    "risultati_geni_unfiltered_ZR75_1_TAM2_treatment_run2.xlsx": "ZR_75_1_TAM2_vs_ZR_75_1wt",
}

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Union, Tuple, Set, List, Any

import numpy as np
import pandas as pd


def integrate_deseq_with_promoter_methylation(
    *,
    deseq_dir: Union[str, Path],
    meth_all_tsv: Union[str, Path],
    map_expr_to_meth: Dict[str, str],
    out_dir: Optional[Union[str, Path]] = None,
    # Filters
    padj_max: float = 0.05,
    basemean_min: Optional[float] = 1,
    abs_log2fc_min: Optional[float] = None,
    keep_meth_status: Set[str] = frozenset({"hyper", "hypo"}),
    # I/O + Excel
    sheet_name: Union[int, str] = 0,
    verbose: bool = True,
) -> Tuple[pd.DataFrame, pd.DataFrame, Path]:
    """
    Integrate DESeq2 gene expression results with gene-level promoter methylation summaries
    and retain directionally coherent genes.

    Coherence definition:
      - promoter_status == 'hyper' AND log2FoldChange < 0  => 'hyper+down'
      - promoter_status == 'hypo'  AND log2FoldChange > 0  => 'hypo+up'
      - otherwise => 'discordant' (discarded from coherent outputs)

    Parameters
    ----------
    deseq_dir : str | Path
        Folder containing DESeq2 result Excel files (.xlsx).
    meth_all_tsv : str | Path
        Path to methylation gene-level promoter table, typically:
        .../gene_level_promoter/ALL_COMPARISONS__gene_promoter.tsv
    map_expr_to_meth : dict[str, str]
        Mapping from DESeq2 Excel filename -> methylation comparison name.
        Example:
            {
              "risultati_geni_unfiltered_MCF7_TamR_treatment_run2.xlsx": "MCF7_TAMR_vs_MCF7WT",
              ...
            }
        The mapping is used to select the correct methylation subset for each expression file.
    out_dir : str | Path | None
        Output directory. If None, defaults to `deseq_dir / "integration_methylation"`.
    padj_max : float, default=0.05
        Keep DE genes with padj < padj_max.
    basemean_min : float | None, default=1
        If not None, keep DE genes with baseMean > basemean_min.
    abs_log2fc_min : float | None, default=None
        If not None, keep DE genes with |log2FoldChange| >= abs_log2fc_min.
    keep_meth_status : set[str], default={"hyper","hypo"}
        Keep only methylation rows whose promoter_status is in this set
        (e.g., exclude "small").
    sheet_name : int | str, default=0
        Excel sheet name or index to read for DESeq2 files.
    verbose : bool, default=True
        If True, prints warnings and output paths similarly to the script.

    Returns
    -------
    coherent_df : pd.DataFrame
        Long table of coherent genes across all mapped comparisons.
    freq_df : pd.DataFrame
        Frequency table of coherent genes across methylation comparisons.
        Empty if coherent_df is empty.
    out_dir_path : Path
        Output directory where TSV files are written.

    Notes
    -----
    - DESeq2 columns expected in each Excel:
        gene_name, baseMean, log2FoldChange, padj
    - Methylation columns expected in meth_all_tsv:
        gene, comparison, promoter_status
      plus extra columns (n_promoter_DMP, min_FDR_promoter, etc.) if present.
    """
    deseq_dir = Path(deseq_dir)
    meth_all_tsv = Path(meth_all_tsv)

    if out_dir is None:
        out_dir = deseq_dir / "integration_methylation"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 
    # Helpers
    # 
    def norm_gene(x: Any) -> str:
        return str(x).strip().upper()

    def read_deseq_xlsx(fp: Path) -> pd.DataFrame:
        df = pd.read_excel(fp, sheet_name=sheet_name, engine="openpyxl")

        needed = {"gene_name", "baseMean", "log2FoldChange", "padj"}
        missing = needed - set(df.columns)
        if missing:
            raise ValueError(f"{fp.name} missing columns: {missing}. Available: {list(df.columns)}")

        df = df[["gene_name", "baseMean", "log2FoldChange", "padj"]].copy()
        df["gene_name"] = df["gene_name"].astype(str).str.strip()
        df["gene_norm"] = df["gene_name"].apply(norm_gene)

        df["baseMean"] = pd.to_numeric(df["baseMean"], errors="coerce")
        df["log2FoldChange"] = pd.to_numeric(df["log2FoldChange"], errors="coerce")
        df["padj"] = pd.to_numeric(df["padj"], errors="coerce")

        df = df.dropna(subset=["gene_norm", "log2FoldChange", "padj"])
        df = df[(df["gene_norm"] != "") & (df["gene_norm"].str.lower() != "nan")]

        # Required filter
        df = df[df["padj"] < padj_max]

        # Optional filters
        if basemean_min is not None:
            df = df[df["baseMean"] > basemean_min]
        if abs_log2fc_min is not None:
            df = df[df["log2FoldChange"].abs() >= abs_log2fc_min]

        if df.empty:
            return df

        # Deduplicate per gene: keep max |log2FC|
        idx = df.groupby("gene_norm")["log2FoldChange"].apply(lambda s: s.abs().idxmax())
        return df.loc[idx].copy()

    # 
    # Load methylation
    # 
    meth = pd.read_csv(meth_all_tsv, sep="\t", low_memory=False)
    if "gene" not in meth.columns or "comparison" not in meth.columns:
        raise ValueError(
            f"Methylation table must contain columns 'gene' and 'comparison'. Columns: {list(meth.columns)}"
        )

    meth["gene_norm"] = meth["gene"].astype(str).str.strip().str.upper()
    meth["comparison"] = meth["comparison"].astype(str).str.strip()

    if "promoter_status" in meth.columns and keep_meth_status:
        meth = meth[meth["promoter_status"].isin(set(keep_meth_status))].copy()

    # 
    # Integration
    # 
    all_rows: List[pd.DataFrame] = []

    for xlsx_name, meth_comp in map_expr_to_meth.items():
        xlsx_path = deseq_dir / xlsx_name
        if not xlsx_path.exists():
            if verbose:
                print(f"Missing DESeq2 file: {xlsx_path}")
            continue

        expr = read_deseq_xlsx(xlsx_path)
        if expr.empty:
            if verbose:
                print(f"No DE genes after filters in: {xlsx_name}")
            continue

        expr["expr_file"] = xlsx_name
        expr["methylation_comparison"] = meth_comp

        msub = meth[meth["comparison"] == meth_comp].copy()
        if msub.empty:
            if verbose:
                print(f"No methylation rows for comparison: {meth_comp}")
            continue

        merged = msub.merge(expr, on="gene_norm", how="inner", suffixes=("_meth", "_expr"))
        if merged.empty:
            continue

        cond_hyper_down = (merged["promoter_status"] == "hyper") & (merged["log2FoldChange"] < 0)
        cond_hypo_up = (merged["promoter_status"] == "hypo") & (merged["log2FoldChange"] > 0)

        merged["coherence"] = np.where(
            cond_hyper_down,
            "hyper+down",
            np.where(cond_hypo_up, "hypo+up", "discordant"),
        )

        coherent = merged[merged["coherence"].isin(["hyper+down", "hypo+up"])].copy()
        if coherent.empty:
            continue

        coherent.insert(0, "expr_dataset", xlsx_path.stem)

        keep_cols = [
            "expr_dataset",
            "expr_file",
            "methylation_comparison",
            "gene_norm",
            "gene",
            "promoter_status",
            "n_promoter_DMP",
            "min_FDR_promoter",
            "max_abs_deltaBeta_promoter",
            "deltaBeta_rep",
            "adjP_rep",
            "log2FoldChange",
            "padj",
            "baseMean",
            "coherence",
        ]
        keep_cols = [c for c in keep_cols if c in coherent.columns]
        coherent = coherent[keep_cols].copy()

        all_rows.append(coherent)

    coherent_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()

    # 
    # Outputs
    # 
    out_main = out_dir / f"COHERENT_GENES_hyperdown_hypoup_padj{padj_max}.tsv"
    coherent_df.to_csv(out_main, sep="\t", index=False)

    if verbose:
        print("Saved:", out_main)
        print("N coherent genes (rows):", coherent_df.shape[0])

    # Frequency summary
    if not coherent_df.empty:
        freq_df = (
            coherent_df.groupby(["gene_norm", "coherence"])
            .agg(
                n_comparisons=("methylation_comparison", "nunique"),
                comparisons=("methylation_comparison", lambda s: ",".join(sorted(set(map(str, s))))),
            )
            .reset_index()
            .sort_values(["n_comparisons", "gene_norm"], ascending=[False, True])
        )

        out_freq = out_dir / f"COHERENT_GENES_frequency_padj{padj_max}.tsv"
        freq_df.to_csv(out_freq, sep="\t", index=False)

        if verbose:
            print("Saved:", out_freq)
    else:
        freq_df = pd.DataFrame()
        if verbose:
            print("No coherent genes found with these filters.")

    return coherent_df, freq_df, out_dirfrom __future__ import annotations



In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

#                             PATH
# Configuration (set paths relative to the repository, or override via env vars)
#                             STRUCT
# Expected repo layout (example):
# repo_root/
#   data/
#     gsea/
#       LeadingEdge_Pos/
#       LeadingEdge_Neg/
#     deseq/
#       InputGSEA/
#     methylation/
#       gene_level_promoter/
#         ALL_COMPARISONS__gene_promoter.tsv
#     integration/
#       integration_methylation_lfc0p5/
#   results/
#     (outputs will be written here)
#
# You can adapt these paths to your repo structure.
REPO_ROOT = Path(".").resolve()

LE_ROOT = REPO_ROOT / "data" / "gsea" / "LeadingEdge_Pos"  # 
BY_PATHWAY = LE_ROOT / "by_pathway"

COH_DIR = REPO_ROOT / "data" / "integration" / "integration_methylation_lfc0p5"

RNK_DIR = REPO_ROOT / "data" / "deseq" / "InputGSEA"
METH_ALL = (
    REPO_ROOT
    / "data"
    / "methylation"
    / "gene_level_promoter"
    / "ALL_COMPARISONS__gene_promoter.tsv"
)

OUT_DIR = REPO_ROOT / "results" / "integration_enrichment_branches" / LE_ROOT.name
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_SUPPORT_LE = 2          # consensus leading-edge threshold
MIN_SUPPORT_BRANCH = 2      # >=2 units for strict/majority
MAJORITY_FRAC = 2 / 3       # majority direction (e.g. 0.66)


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        q[i] = prev
    out = np.empty(n, dtype=float)
    out[order] = q
    return out


def fisher_greater(a, b, c, d):
    """
    Fisher exact one-sided (greater) without scipy:
    hypergeometric tail P(X >= a)
    """
    N = a + b + c + d
    K = a + c
    n = a + b

    from math import lgamma, exp

    def logC(nn, kk):
        if kk < 0 or kk > nn:
            return -np.inf
        return lgamma(nn + 1) - lgamma(kk + 1) - lgamma(nn - kk + 1)

    denom = logC(N, n)
    max_x = min(K, n)

    log_probs = []
    for x in range(a, max_x + 1):
        lp = logC(K, x) + logC(N - K, n - x) - denom
        log_probs.append(lp)

    m = max(log_probs) if log_probs else -np.inf
    p = sum(exp(lp - m) for lp in log_probs) * exp(m) if log_probs else 1.0
    return float(min(1.0, max(0.0, p)))


def load_rnk_genes(rnk_dir: Path):
    genes = set()
    files = sorted(rnk_dir.glob("*.rnk"))
    if not files:
        raise FileNotFoundError(f"No .rnk files found in {rnk_dir}")
    for fp in files:
        df = pd.read_csv(fp, sep="\t", header=None, names=["gene", "score"])
        g = df["gene"].dropna().astype(str).str.strip().str.upper()
        genes.update(g[g != ""].tolist())
    return genes


def load_meth_genes(meth_all: Path):
    df = pd.read_csv(meth_all, sep="\t", low_memory=False)
    if "gene" not in df.columns:
        raise ValueError(
            f"{meth_all.name}: missing required column 'gene'. Columns: {list(df.columns)}"
        )
    return set(df["gene"].dropna().astype(str).str.strip().str.upper())


def load_consensus_leading(by_pathway: Path, min_support: int):
    files = sorted(by_pathway.glob("*__leading_edge_gene_frequency.tsv"))
    if not files:
        raise FileNotFoundError(f"No frequency files found in {by_pathway}")

    le = {}
    for fp in files:
        pathway = fp.name.replace("__leading_edge_gene_frequency.tsv", "")
        df = pd.read_csv(fp, sep="\t", low_memory=False)
        if not {"gene", "n_datasets"}.issubset(df.columns):
            raise ValueError(
                f"{fp.name}: expected columns 'gene' and 'n_datasets' not found. "
                f"Columns: {list(df.columns)}"
            )

        df["n_datasets"] = pd.to_numeric(df["n_datasets"], errors="coerce")
        sub = df[df["n_datasets"] >= min_support].copy()
        genes = set(sub["gene"].dropna().astype(str).str.strip().str.upper())
        le[pathway] = genes
    return le


def load_coherence_df(coh_dir: Path):
    # Prefer COHERENT_GENES_* if multiple exist; pick newest by mtime
    files = sorted(coh_dir.glob("COHERENT_GENES_hyperdown_hypoup_padj*.tsv"))
    if not files:
        raise FileNotFoundError(f"No COHERENT_GENES_*.tsv files found in {coh_dir}")
    coh_path = max(files, key=lambda p: p.stat().st_mtime)

    df = pd.read_csv(coh_path, sep="\t", low_memory=False)
    needed = {"gene_norm", "coherence", "methylation_comparison"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(
            f"{coh_path.name}: missing required columns {missing}. Columns: {list(df.columns)}"
        )

    df["gene_norm"] = df["gene_norm"].astype(str).str.strip().str.upper()
    df["coherence"] = df["coherence"].astype(str).str.strip()
    df["methylation_comparison"] = df["methylation_comparison"].astype(str).str.strip()

    # Keep only coherent (exclude discordant)
    df = df[df["coherence"].isin(["hyper+down", "hypo+up"])].copy()
    return df, coh_path


def build_branch_gene_sets(
    coh_df: pd.DataFrame,
    min_support: int = 2,
    majority_frac: float = 2 / 3,
    unit_col: str = "methylation_comparison",
):
    """
    unit_col = what you treat as a "unit" (here: methylation comparison, i.e. cell line-specific).
    Output:
      - gene_summary: per-gene summary table with counts
      - gene_sets: dict of gene sets split by branch and direction
    """
    # Safety: use unique (gene, unit) per direction
    sub = coh_df[["gene_norm", "coherence", unit_col]].dropna().copy()
    sub = sub.drop_duplicates()

    # Counts per gene and direction
    pivot = (
        sub.groupby(["gene_norm", "coherence"])[unit_col]
        .nunique()
        .unstack(fill_value=0)
    )

    for col in ["hyper+down", "hypo+up"]:
        if col not in pivot.columns:
            pivot[col] = 0

    pivot = pivot.reset_index().rename(columns={"hyper+down": "n_hyperdown", "hypo+up": "n_hypoup"})
    pivot["n_total"] = pivot["n_hyperdown"] + pivot["n_hypoup"]

    # No oppositions = one of the two directions is 0
    pivot["non_divergent"] = (pivot["n_total"] >= 1) & (
        (pivot["n_hyperdown"] == 0) | (pivot["n_hypoup"] == 0)
    )

    # Strict = >=min_support and no opposition
    pivot["strict"] = (pivot["n_total"] >= min_support) & pivot["non_divergent"]

    # Majority = >=min_support and dominant direction fraction >= majority_frac
    pivot["dominant_dir"] = np.where(
        pivot["n_hyperdown"] > pivot["n_hypoup"],
        "hyper+down",
        np.where(pivot["n_hypoup"] > pivot["n_hyperdown"], "hypo+up", "tie"),
    )
    pivot["dominant_n"] = pivot[["n_hyperdown", "n_hypoup"]].max(axis=1)
    pivot["dominant_frac"] = np.where(
        pivot["n_total"] > 0, pivot["dominant_n"] / pivot["n_total"], np.nan
    )
    pivot["majority"] = (
        (pivot["n_total"] >= min_support)
        & (pivot["dominant_dir"] != "tie")
        & (pivot["dominant_frac"] >= majority_frac)
    )

    gene_sets = {}

    # NON-DIVERGENT
    nd = pivot[pivot["non_divergent"]].copy()
    gene_sets["nondivergent_both"] = set(nd["gene_norm"])
    gene_sets["nondivergent_hyperdown"] = set(nd.loc[nd["n_hyperdown"] >= 1, "gene_norm"])
    gene_sets["nondivergent_hypoup"] = set(nd.loc[nd["n_hypoup"] >= 1, "gene_norm"])

    # STRICT
    st = pivot[pivot["strict"]].copy()
    gene_sets["strict_both"] = set(st["gene_norm"])
    gene_sets["strict_hyperdown"] = set(st.loc[st["n_hyperdown"] >= min_support, "gene_norm"])
    gene_sets["strict_hypoup"] = set(st.loc[st["n_hypoup"] >= min_support, "gene_norm"])

    # MAJORITY
    mj = pivot[pivot["majority"]].copy()
    gene_sets["majority_both"] = set(mj["gene_norm"])
    gene_sets["majority_hyperdown"] = set(
        mj.loc[mj["dominant_dir"] == "hyper+down", "gene_norm"]
    )
    gene_sets["majority_hypoup"] = set(
        mj.loc[mj["dominant_dir"] == "hypo+up", "gene_norm"]
    )

    return pivot, gene_sets


def run_fisher_for_branch(le_by_pathway: dict, U: set, coh_set: set):
    coh_set = set(g for g in coh_set if g in U)

    rows = []
    for pw, A0 in le_by_pathway.items():
        A = set(g for g in A0 if g in U)
        if not A:
            continue

        a = len(A & coh_set)
        b = len(A) - a
        c = len(coh_set) - a
        d = len(U) - (a + b + c)

        p = fisher_greater(a, b, c, d)
        orr = ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))

        rows.append(
            {
                "pathway": pw,
                "n_leading_consensus": len(A),
                "n_coherent_in_pathway": a,
                "n_coherent_total": len(coh_set),
                "universe_size": len(U),
                "odds_ratio": float(orr),
                "pvalue": float(p),
                "coherent_genes_in_pathway": ",".join(sorted(A & coh_set)) if a > 0 else "",
            }
        )

    res = pd.DataFrame(rows)
    if res.empty:
        return res

    res["FDR_BH"] = bh_fdr(res["pvalue"].values)
    res = res.sort_values(["FDR_BH", "pvalue", "n_coherent_in_pathway"], ascending=[True, True, False])
    return res


# 
# Run
expr_genes = load_rnk_genes(RNK_DIR)
meth_genes = load_meth_genes(METH_ALL)
U = expr_genes & meth_genes
print(f"Universe U: {len(U)} (RNK ∩ promoter methyl genes)")

le_by_pathway = load_consensus_leading(BY_PATHWAY, MIN_SUPPORT_LE)
print(f"Pathways with leading-edge consensus: {len(le_by_pathway)}")

coh_df, coh_path = load_coherence_df(COH_DIR)
print(f"Coherence file used: {coh_path.name} | coherent rows: {coh_df.shape[0]}")

gene_summary, gene_sets = build_branch_gene_sets(
    coh_df,
    min_support=MIN_SUPPORT_BRANCH,
    majority_frac=MAJORITY_FRAC,
    unit_col="methylation_comparison",
)

gene_summary_out = OUT_DIR / f"BRANCH_gene_summary_min{MIN_SUPPORT_BRANCH}_maj{MAJORITY_FRAC:.2f}.tsv"
gene_summary.to_csv(gene_summary_out, sep="\t", index=False)
print("Saved:", gene_summary_out)

geneset_dir = OUT_DIR / "gene_sets"
geneset_dir.mkdir(exist_ok=True)
for name, gs in gene_sets.items():
    (geneset_dir / f"{name}.txt").write_text(
        "\n".join(sorted(gs)) + ("\n" if gs else ""),
        encoding="utf-8",
    )
print("Saved gene sets in:", geneset_dir)

results_dir = OUT_DIR / "fisher_results"
results_dir.mkdir(exist_ok=True)

for branch_name, gs in gene_sets.items():
    gsU = set(g for g in gs if g in U)
    res = run_fisher_for_branch(le_by_pathway, U, gsU)

    out = results_dir / f"FISHER_{LE_ROOT.name}__{branch_name}__minLE{MIN_SUPPORT_LE}.tsv"
    res.to_csv(out, sep="\t", index=False)
    print(f"{branch_name}: {res.shape[0]} pathways | wrote: {out.name}")

print("\nDONE")
print("You can find:")
print(" - gene summary:", gene_summary_out)
print(" - gene sets:   ", geneset_dir)
print(" - Fisher TSVs: ", results_dir)